### Dataset and Task Metadata

In [1]:
from data_foundry.schema import DatasetMetadata, PredictiveMLTaskMetadata

dataset_mold = DatasetMetadata(
    unique_name="healthcare_insurance_expenses",
    dataset_year="2023",
    domain_str="medical & healthcare",
    # Data Source
    dataset_source="Kaggle",
    original_dataset_source_download_link="https://www.kaggle.com/datasets/arunjangir245/healthcare-insurance-expenses/",
    download_description="""
kaggle datasets download -d arunjangir245/healthcare-insurance-expenses -p local-data-warehouse/healthcare_insurance_expenses/ --unzip
""",
    # References
    academic_reference_bibtex=r"""@misc{arunjangir2452023insurance,
  author       = {Kaggle User Arunjangir245},
  title        = {Healthcare Insurance Expenses},
  year         = {2023},
  howpublished = {\url{https://www.kaggle.com/datasets/arunjangir245/healthcare-insurance-expenses/}},
  note         = {Kaggle dataset},
}
""",
    academic_reference_bibtex_key="arunjangir2452023insurance",
    license="Database Contents License (DbCL) v1.0",
    data_tags=["IID"],
    curation_comments="""
- Unlike in TabArena, we log scale the target as the distribution is very skewed. This is a common practice for price-related targets.
""",
)
task_mold = PredictiveMLTaskMetadata(
    target_column_name="charges",
    problem_type="regression",
    objective_metric_name="rmse",
)

## Preprocessing

In [2]:
import pandas as pd
import numpy as np

df = pd.read_csv(f"{dataset_mold.path}/insurance.csv")

cat_features = [
    "sex",
    "smoker",
    "region",
]

df[cat_features] = df[cat_features].astype("category")

df[task_mold.target_column_name] = np.log(df[task_mold.target_column_name])

df = df.sample(frac=1, random_state=42).reset_index(drop=True)

## Data Checks

In [3]:
from data_foundry import dataset_checks
df_head, summary, numeric_stats, cat_stats, target_df = dataset_checks.run_all_checks(
    data=df,
    classification=task_mold.is_classification,
    target_feature=task_mold.target_column_name,
    print_report=False, # In notebook...
)


#### Dataset Overview
Rows: 1,338
Columns: 7
Use sampling: False (sample size: 1,338)
Get row duplicates (staged, merged)...
Using top-6 columns for initial filtering: ['bmi', 'age', 'children', 'region', 'smoker', 'sex']
Rows remaining as candidates after top-6 filter: 6 (of 1,338)

#### Duplicate Report
Total duplicate rows: 1 (0.07% of dataset)
Duplicate rows ignoring target: 3 (0.22% of dataset)
Get column duplicates...
Duplicate columns: 0 (0.00% of columns)

Data quality checks completed.


In [4]:
# Sample Rows
df_head

,age,sex,bmi,children,smoker,region,charges
0,45,female,25.175,2,no,northeast,9.115488
1,36,female,30.020,0,no,northwest,8.570198
2,64,female,26.885,0,yes,northwest,10.286400
3,46,male,25.745,3,no,northwest,9.137973
4,19,male,31.920,0,yes,northwest,10.426744


In [5]:
# Feature Summary
summary

,index,dtype,n_missing,pct_missing,n_unique,examples
0,sex,category,0.0,0.0,2.0,"male, female"
1,smoker,category,0.0,0.0,2.0,"no, yes"
2,region,category,0.0,0.0,4.0,"southeast, northwest, southwest, northeast"
3,bmi,float64,0.0,0.0,548.0,"32.3, 28.31, 31.35, 34.1, 28.88, 30.8, 30.875, 30.495, 33.33, 38.06"
4,charges,float64,0.0,0.0,1337.0,"7.4022, 9.1026, 9.4776, 9.8909, 8.8452, 9.5553, 7.9553, 10.4576, 10.7408, 9.8124"
5,age,int64,0.0,0.0,47.0,"18, 19, 46, 47, 52, 50, 45, 20, 48, 51"
6,children,int64,0.0,0.0,6.0,"0, 1, 2, 3, 4, 5"


In [6]:
# Numeric Feature Statistics
numeric_stats

,count,mean,std,min,max
age,1338.0,39.207025,14.049960,18.000000,64.000000
bmi,1338.0,30.663397,6.098187,15.960000,53.130000
children,1338.0,1.094918,1.205493,0.000000,5.000000
charges,1338.0,9.098659,0.919527,7.022756,11.063045


In [7]:
# Categorical Feature Statistics
cat_stats

value  count    pct
column rank                         
region 1     southeast    364  27.20
       2     northwest    325  24.29
       3     southwest    325  24.29
       4     northeast    324  24.22
sex    1          male    676  50.52
       2        female    662  49.48
smoker 1            no   1064  79.52
       2           yes    274  20.48

In [8]:
# Target Distribution
target_df

,y_missing_count,non_positive_pct,skew_y,skew_log,var_y,var_log,log_used,aic_exponential,aic_lognormal,dist_hint
0,0,0.0,-0.09,-0.3,0.846,0.011,log,13685.4,4.750286e+15,exponential


## Task Curation

In [9]:
from data_foundry.curation_recommendations import get_recommended_splits_dimensions

n_repeats, n_splits, none_or_test_size = get_recommended_splits_dimensions(dataset=df)
print(f"Recommended IID splits: n_repeats={n_repeats}, n_splits={n_splits}, test_size={none_or_test_size}")

Recommended IID splits: n_repeats=10, n_splits=3, test_size=None


In [10]:
from data_foundry.schema import PredictiveMLSplitsMetadata
from data_foundry.curation_recommendations import get_recommended_iid_splits

splits_mold = PredictiveMLSplitsMetadata(
    splits_comment="Default splits for IID data.",
    splits=get_recommended_iid_splits(
        dataset=df,
        n_repeats=n_repeats,
        n_splits=n_splits,
        test_size=none_or_test_size,
        stratify_on=task_mold.stratify_on,
    ),
)

## Export

In [11]:
from data_foundry.curation_container import CuratedContainer
curated_data = CuratedContainer(
    dataset=df,
    dataset_metadata=dataset_mold,
    task_metadata=task_mold,
    experiment_metadata=splits_mold,
 )
curated_data.save()
print(curated_data.uuid)
print(curated_data.checksum)

Calculating checksum for curated container...
Saving curated container to healthcare_insurance_expenses/019d67b9-c865-77be-a4a2-c2d2a1afe39a
019d67b9-c865-77be-a4a2-c2d2a1afe39a
d195c5516becf93715916db32a445524c73c21b7c4bebd1db687bd33e3585776
